### What data augmentation really is

Data augmentation means

- creating new training samples
- by slightly modifying existing images
- without changing their labels

Examples
- flip
- crop
- shift
- rotate (small amounts)

Key idea
- the model sees many variations of the same object
- so it learns concepts, not pixel positions

### Important rule

Augmentation is applied

- only to training data
- never to test data

Why
- test data must represent reality
- augmentation during testing corrupts evaluation

So we define two transforms.

### Code

In [1]:
from torchvision import datasets, transforms

# transform = transforms.ToTensor()

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor()
])

`Compose()`
- chains multiple transforms in order

`RandomHorizontalFlip()`
- flips image left–right with probability 0.5
- teaches symmetry

`RandomCrop(32, padding=4)`
- pads image to 40×40
- randomly crops back to 32×32
- simulates small shifts

`ToTensor()`
- converts to tensor
- rescales to [0, 1]

In [2]:
test_transform = transforms.ToTensor()

In [3]:
train_data = datasets.CIFAR10(
    root="data",
    download=False,
    train=True,
    transform=train_transform
)

test_data = datasets.CIFAR10(
    root="data",
    download=False,
    train=False,
    transform=test_transform
)

In [4]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

In [5]:
import torch.nn as nn

model = nn.Sequential(
    nn.Conv2d(3, 32, kernel_size=3, padding=1),
    nn.BatchNorm2d(32),
    nn.ReLU(),
    
    nn.Conv2d(32, 64, kernel_size=3, padding=1),
    nn.BatchNorm2d(64),
    nn.ReLU(),
    
    nn.MaxPool2d(2),
    
    nn.Conv2d(64, 128, kernel_size=3, padding=1),
    nn.BatchNorm2d(128),
    nn.ReLU(),
    
    nn.MaxPool2d(2),
    
    nn.Flatten(),
    nn.Linear(128*8*8, 10)
)

In [6]:
import torch

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

In [7]:
epochs = 10

for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        preds = model(images)
        loss = loss_fn(preds, labels)
        
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
        epoch_loss += loss.item()
        
    epoch_loss /= len(train_loader)
    
    print(f"Epoch: {epoch} | Loss per epoch: {epoch_loss:.4f}")      
        

Epoch: 0 | Loss: 2.5580
Epoch: 0 | Loss: 5.8496
Epoch: 0 | Loss: 4.3924
Epoch: 0 | Loss: 5.0535
Epoch: 0 | Loss: 4.9326
Epoch: 0 | Loss: 3.2400
Epoch: 0 | Loss: 3.3076
Epoch: 0 | Loss: 3.7177
Epoch: 0 | Loss: 3.6604
Epoch: 0 | Loss: 3.1157
Epoch: 0 | Loss: 2.6899
Epoch: 0 | Loss: 3.6129
Epoch: 0 | Loss: 3.6540
Epoch: 0 | Loss: 3.0210
Epoch: 0 | Loss: 2.9468
Epoch: 0 | Loss: 2.6164
Epoch: 0 | Loss: 2.4300
Epoch: 0 | Loss: 2.2785
Epoch: 0 | Loss: 2.6917
Epoch: 0 | Loss: 2.3693
Epoch: 0 | Loss: 2.1920
Epoch: 0 | Loss: 2.1728
Epoch: 0 | Loss: 2.2966
Epoch: 0 | Loss: 2.3737
Epoch: 0 | Loss: 2.1515
Epoch: 0 | Loss: 1.9980
Epoch: 0 | Loss: 2.1918
Epoch: 0 | Loss: 1.8969
Epoch: 0 | Loss: 2.1537
Epoch: 0 | Loss: 2.1239
Epoch: 0 | Loss: 2.1470
Epoch: 0 | Loss: 2.1280
Epoch: 0 | Loss: 2.0005
Epoch: 0 | Loss: 1.8606
Epoch: 0 | Loss: 2.1313
Epoch: 0 | Loss: 2.0591
Epoch: 0 | Loss: 1.8980
Epoch: 0 | Loss: 2.0488
Epoch: 0 | Loss: 2.0340
Epoch: 0 | Loss: 2.2831
Epoch: 0 | Loss: 2.1746
Epoch: 0 | Loss:

In [8]:
import torch.nn.functional as F

def shift_right(images, pixels=2):
    return F.pad(images, (pixels, 0, 0, 0))[:, :, :, :-pixels]

In [9]:
def evaluate(transform_fn):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels  = labels.to(device)
            
            images = transform_fn(images)
            preds = model(images)
            predicted = preds.argmax(dim=1)
            
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
            
        accuracy = correct / total
    print(accuracy)

In [10]:
evaluate(lambda x: x)   #Clean test accuracy
evaluate(shift_right)   #Shifted test accuracy

0.7763
0.7659


similar accuracy -> model is trained correctly and works fine on augmented data